# RunPod WAN 2.6 Text-to-Image\nSubmits a prompt, polls for completion, and downloads the generated image.

In [ ]:
## Cell 1 — Install dependencies (run once)
# %pip install requests python-dotenv Pillow

In [1]:
import os, time, base64, requests
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage
from PIL import Image
import io

load_dotenv()
API_KEY = os.getenv("RUNPOD_API")
assert API_KEY, "RUNPOD_API not found in .env"
print("API key loaded ✓")

API key loaded ✓


In [2]:
## ── Config ──────────────────────────────────────────────────────────────────
ENDPOINT   = "https://api.runpod.ai/v2/wan-2-6-t2i"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

PAYLOAD = {
    "input": {
        "prompt": (
            "A modern tea shop interior, warm afternoon light, minimalist wood design, "
            "cinematic photography, medium shot, shallow depth of field, 35mm look, "
            "clean lines, natural shadows, soft highlights, cozy seating, neatly arranged "
            "tea bar, high detail, "
            "Negative prompt: blurry, low-res, watermark, text, logo, cluttered background, "
            "overexposed, underexposed, distortion, fisheye, noise"
        ),
        "size": "1024*1024",
        "seed": -1,
        "enable_safety_checker": True,
        "width": None,
        "height": None,
    }
}
print("Config ready ✓")

Config ready ✓


In [3]:
## ── Submit job ───────────────────────────────────────────────────────────────
resp = requests.post(f"{ENDPOINT}/run", headers=HEADERS, json=PAYLOAD)
resp.raise_for_status()
job = resp.json()
job_id = job["id"]
print(f"Job submitted → id: {job_id}  status: {job.get('status')}")

Job submitted → id: 07b39cad-d6b1-4744-b324-309160589c96-e1  status: IN_QUEUE


In [4]:
## ── Poll until complete ──────────────────────────────────────────────────────
POLL_INTERVAL = 5   # seconds between checks
MAX_WAIT      = 600  # give up after 10 min

start = time.time()
while True:
    elapsed = int(time.time() - start)
    status_resp = requests.get(f"{ENDPOINT}/status/{job_id}", headers=HEADERS)
    status_resp.raise_for_status()
    result = status_resp.json()
    status = result.get("status")
    print(f"[{elapsed:>4}s] status: {status}")

    if status == "COMPLETED":
        print("Done!")
        break
    elif status in ("FAILED", "CANCELLED", "TIMED_OUT"):
        raise RuntimeError(f"Job ended with status: {status}\n{result}")
    elif elapsed > MAX_WAIT:
        raise TimeoutError(f"Gave up after {MAX_WAIT}s")

    time.sleep(POLL_INTERVAL)

[   0s] status: IN_PROGRESS
[   5s] status: IN_PROGRESS
[  14s] status: COMPLETED
Done!


In [9]:
import requests

url = result['output']['result']
response = requests.get(url)

with open('output.png', 'wb') as f:
    f.write(response.content)
